## Training of Computer Vision Model

In [ ]:
!pip install ultralytics

In [ ]:
# Standard library imports
import os
import time
import copy
from pathlib import Path
from collections import defaultdict

# Third-party imports
import cv2
import torch
import numpy as np
from PIL import Image
from ultralytics import YOLO

# Torchvision specific imports
import torchvision
from torchvision.transforms import functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.ops import box_iou
from torch.utils.data import Dataset, DataLoader

### YOLO 11m Model Training

In [ ]:


model = YOLO("yolo11m.pt")

results = model.train(
    data=r"C:\Users\HP Victus\CDS_2026Spring_project\final_unified_dataset\dataset.yaml",
    
    #  Training duration 
    epochs=150,          # was 100 — bag recall needs more time to converge
    patience=40,         # was 30 — give it more room before early stopping

    # Batch / hardware 
    imgsz=640,           # keeping original — safe for your GPU
    batch=16,
    device="0",
    workers=4,
    amp=True,
    seed=0,

    # Class imbalance fix 
    cls=1.5,             # upweight classification loss — penalises
                         # missing the minority bag class more strongly

    #  Box localisation (mAP50-95 gap) 
    box=9.0,             # tighter box regression — addresses the
                         # ~20pp mAP50→mAP50-95 drop

    #  NMS (duplicate box fix) 
    iou=0.4,             # stricter NMS suppresses the duplicate
                         # overlapping boxes seen in the video

    # Augmentation — compensates for no resolution bump
    scale=0.6,           # wider scale range catches small/distant objects
    copy_paste=0.3,      # bumped from 0.2 — pastes bag instances onto new
                         # backgrounds, directly boosts effective bag samples
    mixup=0.15,          # blends two images — exposes model to bags
                         # overlapping people in unusual ways
    degrees=10.0,        # mild rotation — bags carried at many angles
    fliplr=0.5,          # horizontal flip (default, explicitly kept)
    mosaic=1.0,          # keep mosaic on (default, explicitly kept)
    close_mosaic=15,     # was 10 — keep mosaic longer before turning it off

    # Output 
    project=r"C:\Users\HP Victus\CDS_2026Spring_project\runs\detect\yolo11m_COCO_V2",
    name="exp2",
    plots=True,
)

print(f"Training complete! Results saved to: {results.save_dir}")

### Fine-tuned model evaluation

In [ ]:



model = YOLO("../../runs/detect/yolo11m_COCO_V2/exp2/weights/best.pt")
metrics = model.val(
    data="../../final_unified_dataset/dataset.yaml",
    imgsz=640,
    split='test',
    plots=True,
)

print(f"Person AP50: {metrics.box.ap50[0]:.3f}")
print(f"Bag    AP50: {metrics.box.ap50[1]:.3f}")
print(f"Person AP50-95: {metrics.box.ap[0]:.3f}")
print(f"Bag    AP50-95: {metrics.box.ap[1]:.3f}")

### Testing Fine-tuned model against webcam

In [ ]:



# Load your best weights
model = YOLO("../../runs/detect/yolo11m_COCO_V2/exp2/weights/best.pt")

# 1. Lower the confidence to 0.1 (10%) to see 'weak' detections
# 2. Set imgsz to 640 (or whatever you used in training)
results = model.predict(source="0", show=True, conf=0.5, imgsz=640)

### Testing Fine-tuned model against created test footages

In [ ]:



# Config 
WEIGHT_PATH = "../../runs/detect/yolo11m_COCO_V2/exp2/weights/best.pt" # path to your trained weights
VIDEO_PATH  = "./test3.mp4"       # path to your test footage
OUTPUT_PATH = "./output3.mp4" # where to save the result
CONF        = 0.5
IMGSZ       = 640

# Load model 
model  = YOLO(WEIGHT_PATH)
cap    = cv2.VideoCapture(VIDEO_PATH)

# Match output video properties to input
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
)

print(f"Processing {total} frames at {fps:.1f} fps...")

# Run inference frame by frame
frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results    = model.predict(source=frame, conf=CONF, verbose=False,imgsz=IMGSZ,iou=0.3)
    annotated  = results[0].plot()

    out.write(annotated)

    # Progress every 50 frames
    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"  {frame_idx}/{total} frames done ({frame_idx/total*100:.1f}%)")

cap.release()
out.release()
print(f"\n✅ Saved annotated video → {OUTPUT_PATH}")

Processing 626 frames at 30.0 fps...
  50/626 frames done (8.0%)
  100/626 frames done (16.0%)
  150/626 frames done (24.0%)
  200/626 frames done (31.9%)
  250/626 frames done (39.9%)
  300/626 frames done (47.9%)
  350/626 frames done (55.9%)
  400/626 frames done (63.9%)
  450/626 frames done (71.9%)
  500/626 frames done (79.9%)
  550/626 frames done (87.9%)
  600/626 frames done (95.8%)

✅ Saved annotated video → ./output3.mp4


### Hyperparameter tuning with validation dataset to find the best confidence

In [ ]:


model = YOLO(r"C:\Users\HP Victus\CDS_2026Spring_project\runs\detect\yolo11m_COCO_V2\exp2\weights\best.pt")

for conf in [0.05, 0.10, 0.15, 0.25]:
    metrics = model.val(
        data=r"C:\Users\HP Victus\CDS_2026Spring_project\new_dataset\dataset.yaml",
        split="val",
        conf=conf,
        iou=0.5,
        plots=False,
    )
    print(f"conf={conf} → P:{metrics.box.mp:.3f} R:{metrics.box.mr:.3f} mAP50:{metrics.box.map50:.3f}")

# Faster R-CNN

### Training

In [ ]:

# =========================
# CONFIG
# =========================
DATASET_ROOT = "/Users/anbu/Sri/CDS_2026Spring_project/final_unified_dataset"
SAVE_DIR = "./fasterrcnn_outputs"

NUM_CLASSES = 3   # background + person + bag
NUM_EPOCHS = 1    # <- only 1 epoch
BATCH_SIZE = 2    # smaller batch is safer on Mac
LR = 0.0005       # lower LR to avoid NaNs
WEIGHT_DECAY = 0.0005
MOMENTUM = 0.9
NUM_WORKERS = 0   # Mac / notebook safe
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]

# Safer on Mac for Faster R-CNN
PREFER_MPS = False   # set True only if you really want to try MPS again

os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# DEVICE
# =========================
if PREFER_MPS and torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# =========================
# HELPERS
# =========================
def collate_fn(batch):
    return tuple(zip(*batch))

def get_image_files(folder):
    folder = Path(folder)
    files = []
    for ext in IMAGE_EXTENSIONS:
        files.extend(folder.glob(f"*{ext}"))
    return sorted(files)

def yolo_to_xyxy(x_center, y_center, w, h, img_w, img_h):
    x_center *= img_w
    y_center *= img_h
    w *= img_w
    h *= img_h

    x1 = x_center - w / 2
    y1 = y_center - h / 2
    x2 = x_center + w / 2
    y2 = y_center + h / 2

    x1 = max(0.0, min(x1, img_w - 1))
    y1 = max(0.0, min(y1, img_h - 1))
    x2 = max(0.0, min(x2, img_w - 1))
    y2 = max(0.0, min(y2, img_h - 1))

    return [x1, y1, x2, y2]

def parse_yolo_label_file(label_path, img_w, img_h):
    boxes = []
    labels = []
    areas = []
    iscrowd = []

    if not label_path.exists():
        return boxes, labels, areas, iscrowd

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            continue

        try:
            cls_id, x_c, y_c, w, h = map(float, parts)
        except ValueError:
            continue

        # basic YOLO sanity checks
        if not (0 <= cls_id <= 1):
            continue
        if not (0.0 <= x_c <= 1.0 and 0.0 <= y_c <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
            continue

        # Faster R-CNN labels: 0 is background, so shift by +1
        label = int(cls_id) + 1

        box = yolo_to_xyxy(x_c, y_c, w, h, img_w, img_h)
        x1, y1, x2, y2 = box

        bw = x2 - x1
        bh = y2 - y1

        # reject invalid or tiny boxes
        if bw <= 1 or bh <= 1:
            continue

        boxes.append(box)
        labels.append(label)
        areas.append(bw * bh)
        iscrowd.append(0)

    return boxes, labels, areas, iscrowd

# =========================
# DATASET
# =========================
class YOLODetectionDataset(Dataset):
    def __init__(self, root, split="train", transforms=None):
        self.root = Path(root)
        self.split = split
        self.transforms = transforms

        self.image_dir = self.root / "images" / split
        self.label_dir = self.root / "labels" / split

        all_image_files = get_image_files(self.image_dir)
        if len(all_image_files) == 0:
            raise ValueError(f"No images found in {self.image_dir}")

        # Keep only images with at least one valid box
        self.image_files = []
        dropped = 0

        for img_path in all_image_files:
            label_path = self.label_dir / f"{img_path.stem}.txt"
            try:
                with Image.open(img_path) as image:
                    img_w, img_h = image.size
                boxes, labels, areas, iscrowd = parse_yolo_label_file(label_path, img_w, img_h)
                if len(boxes) > 0:
                    self.image_files.append(img_path)
                else:
                    dropped += 1
            except Exception:
                dropped += 1

        if len(self.image_files) == 0:
            raise ValueError(f"No valid labeled images found in {self.image_dir}")

        print(f"[{split}] Kept {len(self.image_files)} valid images, dropped {dropped} invalid/empty images")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        label_path = self.label_dir / f"{img_path.stem}.txt"

        image = Image.open(img_path).convert("RGB")
        img_w, img_h = image.size

        boxes, labels, areas, iscrowd = parse_yolo_label_file(label_path, img_w, img_h)

        # Since invalid/empty files were filtered in __init__, this should be safe.
        # But keep a defensive fallback.
        if len(boxes) == 0:
            boxes = torch.tensor([[0.0, 0.0, 2.0, 2.0]], dtype=torch.float32)
            labels = torch.tensor([1], dtype=torch.int64)
            areas = torch.tensor([4.0], dtype=torch.float32)
            iscrowd = torch.tensor([0], dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)

        image_id = torch.tensor([idx], dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": areas,
            "iscrowd": iscrowd
        }

        image = F.to_tensor(image)

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target

# =========================
# MODEL
# =========================
def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
        in_features, num_classes
    )
    return model

# =========================
# TRAIN / EVAL
# =========================
def train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=20):
    model.train()
    running_loss = 0.0
    valid_steps = 0

    for i, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        try:
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            if not torch.isfinite(losses):
                loss_items = {k: float(v.item()) for k, v in loss_dict.items()}
                print(f"[WARNING] Non-finite loss at step {i}. Skipping batch. Losses: {loss_items}")
                optimizer.zero_grad(set_to_none=True)
                continue

            optimizer.zero_grad(set_to_none=True)
            losses.backward()

            # gradient clipping helps stop exploding updates
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()

            loss_value = float(losses.item())
            running_loss += loss_value
            valid_steps += 1

            if i % print_freq == 0:
                loss_items = {k: float(v.item()) for k, v in loss_dict.items()}
                print(
                    f"Epoch [{epoch}] Step [{i}/{len(data_loader)}] "
                    f"Total Loss: {loss_value:.4f} | {loss_items}"
                )

        except RuntimeError as e:
            print(f"[WARNING] Runtime error at step {i}. Skipping batch. Error: {e}")
            optimizer.zero_grad(set_to_none=True)
            continue

    if valid_steps == 0:
        return float("inf")
    return running_loss / valid_steps

@torch.no_grad()
def evaluate_loss(model, data_loader, device):
    model.train()  # needed for detection loss computation
    total_loss = 0.0
    valid_steps = 0

    for i, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        try:
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            if not torch.isfinite(losses):
                print(f"[WARNING] Non-finite val loss at step {i}. Skipping batch.")
                continue

            total_loss += float(losses.item())
            valid_steps += 1

        except RuntimeError as e:
            print(f"[WARNING] Runtime error during validation at step {i}. Skipping batch. Error: {e}")
            continue

    model.eval()

    if valid_steps == 0:
        return float("inf")
    return total_loss / valid_steps

# =========================
# MAIN
# =========================
def main():
    train_dataset = YOLODetectionDataset(DATASET_ROOT, split="train")
    val_dataset   = YOLODetectionDataset(DATASET_ROOT, split="val")
    test_dataset  = YOLODetectionDataset(DATASET_ROOT, split="test")

    print(f"Train size: {len(train_dataset)}")
    print(f"Val size:   {len(val_dataset)}")
    print(f"Test size:  {len(test_dataset)}")

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn
    )

    model = get_model(NUM_CLASSES)
    model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(
        params,
        lr=LR,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )

    lr_scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=5,
        gamma=0.1
    )

    best_val_loss = float("inf")
    best_model_wts = copy.deepcopy(model.state_dict())

    history = {
        "train_loss": [],
        "val_loss": []
    }

    print("\nStarting training...\n")
    start_time = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start = time.time()

        train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
        val_loss = evaluate_loss(model, val_loader, device)

        lr_scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss:   {val_loss:.4f}")
        print(f"Epoch Time: {time.time() - epoch_start:.2f}s")

        last_ckpt = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "history": history
        }
        torch.save(last_ckpt, os.path.join(SAVE_DIR, "last_fasterrcnn.pth"))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())

            best_ckpt = {
                "epoch": epoch,
                "model_state_dict": best_model_wts,
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss,
                "history": history
            }
            torch.save(best_ckpt, os.path.join(SAVE_DIR, "best_fasterrcnn.pth"))
            print("Best model updated.")

        print("-" * 60)

    total_time = time.time() - start_time
    print(f"Training completed in {total_time / 60:.2f} minutes")

    model.load_state_dict(best_model_wts)
    model.eval()

    print("\nBest validation loss:", best_val_loss)
    print("Saved:")
    print(os.path.join(SAVE_DIR, "best_fasterrcnn.pth"))
    print(os.path.join(SAVE_DIR, "last_fasterrcnn.pth"))

    print("\nRunning quick test inference sanity check...")
    with torch.no_grad():
        for images, targets in test_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            print(f"Batch size: {len(outputs)}")
            for i, output in enumerate(outputs[:2]):
                print(f"Image {i}:")
                print("Boxes:", output["boxes"][:5].cpu())
                print("Labels:", output["labels"][:5].cpu())
                print("Scores:", output["scores"][:5].cpu())
            break

if __name__ == "__main__":
    main()

Using device: cpu
[train] Kept 4038 valid images, dropped 0 invalid/empty images
[val] Kept 504 valid images, dropped 0 invalid/empty images
[test] Kept 507 valid images, dropped 0 invalid/empty images
Train size: 4038
Val size:   504
Test size:  507

Starting training...

Epoch [1] Step [0/2019] Total Loss: 1.5040 | {'loss_classifier': 0.9623014330863953, 'loss_box_reg': 0.5198880434036255, 'loss_objectness': 0.010588468983769417, 'loss_rpn_box_reg': 0.011257600970566273}
Epoch [1] Step [20/2019] Total Loss: 1.3027 | {'loss_classifier': 0.5192515850067139, 'loss_box_reg': 0.705781877040863, 'loss_objectness': 0.04230562970042229, 'loss_rpn_box_reg': 0.035410668700933456}
Epoch [1] Step [40/2019] Total Loss: 0.4805 | {'loss_classifier': 0.16315513849258423, 'loss_box_reg': 0.2854793071746826, 'loss_objectness': 0.029140224680304527, 'loss_rpn_box_reg': 0.002716017421334982}
Epoch [1] Step [60/2019] Total Loss: 0.6434 | {'loss_classifier': 0.1989731341600418, 'loss_box_reg': 0.372487485

### Evaluation of Faster R-CNN

In [ ]:


# ----------------------------
# CONFIG
# ----------------------------
CKPT_PATH = "./fasterrcnn_outputs/best_fasterrcnn.pth"
IOU_THRESHOLDS = np.arange(0.50, 0.96, 0.05)
SCORE_THRESH_FOR_PR = 0.05   # keep low for eval
MAX_DETS = 100

# ----------------------------
# Recreate model and test loader
# Assumes these already exist in earlier cells:
# - NUM_CLASSES
# - device
# - collate_fn
# - YOLODetectionDataset
# - DATASET_ROOT
# - get_model()
# ----------------------------

test_dataset = YOLODetectionDataset(DATASET_ROOT, split="test")
test_loader = DataLoader(
    test_dataset,
    batch_size=1,   # best for clean timing/eval
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

model = get_model(NUM_CLASSES)
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()

print(f"Loaded checkpoint: {CKPT_PATH}")
print(f"Evaluating on {len(test_dataset)} test images...")

# ----------------------------
# Helpers
# ----------------------------
def compute_ap(recall, precision):
    """
    COCO-style AP integration from precision-recall curve.
    """
    recall = np.asarray(recall)
    precision = np.asarray(precision)

    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))

    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])

    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])
    return ap

def match_predictions_to_gt(pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, iou_thr):
    """
    Greedy one-to-one matching per class for one image at a specific IoU threshold.
    Returns:
      pred_records: list of (score, is_tp, pred_label)
      gt_count_per_class: dict[class_id] = number of GT objects
    """
    pred_records = []
    gt_count_per_class = defaultdict(int)

    unique_classes = set(gt_labels.tolist()) | set(pred_labels.tolist())

    for cls in unique_classes:
        gt_mask = (gt_labels == cls)
        pr_mask = (pred_labels == cls)

        cls_gt_boxes = gt_boxes[gt_mask]
        cls_pred_boxes = pred_boxes[pr_mask]
        cls_pred_scores = pred_scores[pr_mask]

        gt_count_per_class[int(cls)] += len(cls_gt_boxes)

        if len(cls_pred_boxes) == 0:
            continue

        order = torch.argsort(cls_pred_scores, descending=True)
        cls_pred_boxes = cls_pred_boxes[order]
        cls_pred_scores = cls_pred_scores[order]

        matched_gt = torch.zeros(len(cls_gt_boxes), dtype=torch.bool)

        if len(cls_gt_boxes) > 0:
            ious = box_iou(cls_pred_boxes, cls_gt_boxes)
        else:
            ious = torch.zeros((len(cls_pred_boxes), 0), dtype=torch.float32)

        for i in range(len(cls_pred_boxes)):
            if ious.shape[1] == 0:
                pred_records.append((float(cls_pred_scores[i]), 0, int(cls)))
                continue

            best_iou, best_gt_idx = ious[i].max(dim=0)
            if best_iou >= iou_thr and not matched_gt[best_gt_idx]:
                matched_gt[best_gt_idx] = True
                pred_records.append((float(cls_pred_scores[i]), 1, int(cls)))
            else:
                pred_records.append((float(cls_pred_scores[i]), 0, int(cls)))

    return pred_records, gt_count_per_class

# ----------------------------
# Run inference once, cache outputs, and time it
# ----------------------------
all_outputs = []
all_targets = []
inference_times = []

with torch.no_grad():
    for images, targets in test_loader:
        images = [img.to(device) for img in images]

        start = time.perf_counter()
        outputs = model(images)
        if device.type == "cuda":
            torch.cuda.synchronize()
        end = time.perf_counter()

        inference_times.append((end - start) * 1000.0)  # ms

        out = outputs[0]
        tgt = targets[0]

        keep = out["scores"].cpu() >= SCORE_THRESH_FOR_PR

        pred_boxes = out["boxes"].detach().cpu()[keep][:MAX_DETS]
        pred_scores = out["scores"].detach().cpu()[keep][:MAX_DETS]
        pred_labels = out["labels"].detach().cpu()[keep][:MAX_DETS]

        gt_boxes = tgt["boxes"].detach().cpu()
        gt_labels = tgt["labels"].detach().cpu()

        all_outputs.append({
            "boxes": pred_boxes,
            "scores": pred_scores,
            "labels": pred_labels,
        })
        all_targets.append({
            "boxes": gt_boxes,
            "labels": gt_labels,
        })

# ----------------------------
# Compute AP / Precision / Recall across IoU thresholds
# ----------------------------
aps_per_iou = []
precisions_per_iou = []
recalls_per_iou = []

for iou_thr in IOU_THRESHOLDS:
    class_pred_records = defaultdict(list)   # class -> [(score, tp), ...]
    class_gt_counts = defaultdict(int)       # class -> gt total

    for out, tgt in zip(all_outputs, all_targets):
        pred_records, gt_count_per_class = match_predictions_to_gt(
            out["boxes"], out["scores"], out["labels"],
            tgt["boxes"], tgt["labels"],
            iou_thr=iou_thr
        )

        for score, is_tp, cls in pred_records:
            class_pred_records[cls].append((score, is_tp))

        for cls, n in gt_count_per_class.items():
            class_gt_counts[cls] += n

    class_aps = []
    class_precisions = []
    class_recalls = []

    eval_classes = sorted(set(class_gt_counts.keys()) | set(class_pred_records.keys()))
    eval_classes = [c for c in eval_classes if c != 0]  # exclude background if present

    for cls in eval_classes:
        preds = class_pred_records[cls]
        n_gt = class_gt_counts[cls]

        if n_gt == 0:
            continue

        if len(preds) == 0:
            class_aps.append(0.0)
            class_precisions.append(0.0)
            class_recalls.append(0.0)
            continue

        preds = sorted(preds, key=lambda x: x[0], reverse=True)
        tps = np.array([p[1] for p in preds], dtype=np.float32)
        fps = 1.0 - tps

        cum_tp = np.cumsum(tps)
        cum_fp = np.cumsum(fps)

        recall_curve = cum_tp / max(n_gt, 1)
        precision_curve = cum_tp / np.maximum(cum_tp + cum_fp, 1e-12)

        ap = compute_ap(recall_curve, precision_curve)
        final_precision = precision_curve[-1] if len(precision_curve) else 0.0
        final_recall = recall_curve[-1] if len(recall_curve) else 0.0

        class_aps.append(ap)
        class_precisions.append(final_precision)
        class_recalls.append(final_recall)

    aps_per_iou.append(np.mean(class_aps) if class_aps else 0.0)
    precisions_per_iou.append(np.mean(class_precisions) if class_precisions else 0.0)
    recalls_per_iou.append(np.mean(class_recalls) if class_recalls else 0.0)

# ----------------------------
# Final metrics
# ----------------------------
map50 = aps_per_iou[0]                        # IoU = 0.50
map50_95 = float(np.mean(aps_per_iou))        # mean over 0.50:0.95
mean_precision = float(np.mean(precisions_per_iou))
mean_recall = float(np.mean(recalls_per_iou))
inference_ms = float(np.mean(inference_times))

print("\n===== Evaluation Results =====")
print(f"mAP50      : {map50:.4f}")
print(f"mAP50-95   : {map50_95:.4f}")
print(f"mAP        : {map50_95:.4f}")   # same as COCO-style mAP
print(f"Precision  : {mean_precision:.4f}")
print(f"Recall     : {mean_recall:.4f}")
print(f"inference_ms/image: {inference_ms:.2f}")

print("\nAP by IoU threshold:")
for thr, ap in zip(IOU_THRESHOLDS, aps_per_iou):
    print(f"IoU {thr:.2f}: AP={ap:.4f}")

[test] Kept 507 valid images, dropped 0 invalid/empty images
Loaded checkpoint: ./fasterrcnn_outputs/best_fasterrcnn.pth
Evaluating on 507 test images...

===== Evaluation Results =====
mAP50      : 0.6189
mAP50-95   : 0.3478
mAP        : 0.3478
Precision  : 0.0840
Recall     : 0.5128
inference_ms/image: 387.56

AP by IoU threshold:
IoU 0.50: AP=0.6189
IoU 0.55: AP=0.5881
IoU 0.60: AP=0.5520
IoU 0.65: AP=0.4981
IoU 0.70: AP=0.4325
IoU 0.75: AP=0.3407
IoU 0.80: AP=0.2546
IoU 0.85: AP=0.1447
IoU 0.90: AP=0.0455
IoU 0.95: AP=0.0027
